In [ ]:
# %% [markdown]
# ## 1. Define helper function
#
# %% code

import json as json_lib
import textwrap
import time

import altair as alt
import IPython
import pandas as pd
import proto
import requests
from google.iam.v1 import iam_policy_pb2
from google.iam.v1 import policy_pb2
from google.protobuf import field_mask_pb2
from google.protobuf.json_format import MessageToDict, MessageToJson
from IPython.display import HTML, display
from pygments import formatters, highlight, lexers


def handle_text_response(resp):
    parts = resp.parts
    full_text = "".join(parts)
    if "\n" not in full_text and len(full_text) > 80:
        wrapped_text = textwrap.fill(full_text, width=80)
        print(wrapped_text)
    else:
        print(full_text)


def display_schema(data):
    fields = getattr(data, "fields")
    df = pd.DataFrame({
        "Column": map(lambda field: getattr(field, "name"), fields),
        "Type": map(lambda field: getattr(field, "type"), fields),
        "Description": map(lambda field: getattr(field, "description", "-"), fields),
        "Mode": map(lambda field: getattr(field, "mode"), fields),
    })
    display(df)


def display_section_title(text):
    display(HTML("<h2>{}</h2>".format(text)))


def format_looker_table_ref(table_ref):
    return "lookmlModel: {}, explore: {}, lookerInstanceUri: {}".format(
        table_ref.lookml_model, table_ref.explore, table_ref.looker_instance_uri
    )


def format_bq_table_ref(table_ref):
    return "{}.{}.{}".format(
        table_ref.project_id, table_ref.dataset_id, table_ref.table_id
    )


def display_datasource(datasource):
    source_name = ""
    if "studio_datasource_id" in datasource:
        source_name = getattr(datasource, "studio_datasource_id")
    elif "looker_explore_reference" in datasource:
        source_name = format_looker_table_ref(
            getattr(datasource, "looker_explore_reference")
        )
    else:
        source_name = format_bq_table_ref(
            getattr(datasource, "bigquery_table_reference")
        )

    print(source_name)
    display_schema(datasource.schema)


def handle_schema_response(resp):
    if "query" in resp:
        print(resp.query.question)
    elif "result" in resp:
        display_section_title("Schema resolved")
        print("Data sources:")
        for datasource in resp.result.datasources:
            display_datasource(datasource)


def handle_data_response(resp):
    if "query" in resp:
        query = resp.query
        display_section_title("Retrieval query")
        print(f"Query name: {query.name}")
        if "question" in query:
            print(f"Question: {query.question}")
        if "datasources" in query:
            print("Data sources:")
            for datasource in query.datasources:
                display_datasource(datasource)
    elif "generated_sql" in resp:
        display_section_title("SQL generated")
        print(resp.generated_sql)
    elif "result" in resp:
        display_section_title("Data retrieved")

        fields = [field.name for field in resp.result.schema.fields]
        d = {}
        for el in resp.result.data:
            for field in fields:
                if field in d:
                    d[field].append(el[field])
                else:
                    d[field] = [el[field]]

        display(pd.DataFrame(d))


def handle_chart_response(resp):
    def _value_to_dict(v):
        if isinstance(v, proto.marshal.collections.maps.MapComposite):
            return _map_to_dict(v)
        elif isinstance(v, proto.marshal.collections.RepeatedComposite):
            return [_value_to_dict(el) for el in v]
        elif isinstance(v, (int, float, str, bool)):
            return v
        else:
            return MessageToDict(v)

    def _map_to_dict(d):
        out = {}
        for k in d:
            if isinstance(d[k], proto.marshal.collections.maps.MapComposite):
                out[k] = _map_to_dict(d[k])
            else:
                out[k] = _value_to_dict(d[k])
        return out

    if "query" in resp:
        print(resp.query.instructions)
    elif "result" in resp:
        vegaConfig = resp.result.vega_config
        vegaConfig_dict = _map_to_dict(vegaConfig)
        alt.Chart.from_json(json_lib.dumps(vegaConfig_dict)).display()


def show_message(msg):
    m = msg.system_message
    if "text" in m:
        handle_text_response(getattr(m, "text"))
    elif "schema" in m:
        handle_schema_response(getattr(m, "schema"))
    elif "data" in m:
        handle_data_response(getattr(m, "data"))
    elif "chart" in m:
        handle_chart_response(getattr(m, "chart"))
    print("\n")


In [ ]:
# %% [markdown]
# ## 2. Authenticate GDA API
#
# %% code

%pip install google-cloud-geminidataanalytics
from google.colab import auth
auth.authenticate_user()

from google.cloud import geminidataanalytics

data_agent_client = geminidataanalytics.DataAgentServiceClient()
data_chat_client = geminidataanalytics.DataChatServiceClient()


In [ ]:
# %% [markdown]
# ## 3. Configure Data Source Details
#
# Update the variables below with your specific Google Cloud project, spanner instance details, and desired agent settings.

# %% code
# Replace with your actual project and spanner details
billing_project = "project"  # Your billing project ID
location = "location"  # The region of your spanner instance

# spanner connection details
spanner_project_id = "project-id"  # Project ID of the spanner instance
engine = "GOOGLE_SQL"
spanner_instance_id = "instance-id"  # Your spanner instance ID
spanner_database_id = "database-id"  # Your database name (e.g., postgres)

system_instruction = "Help the user analyze data from the spanner database"

# Data Agent ID
data_agent_id = "spanner_agent_e2e_example"

In [ ]:
# %% [markdown]
# ## 4. Connect to spanner source
#
# Update the variables below with your specific Google Cloud project, spanner instance details, and desired agent settings.
# # Its better to provide context_set_id similar to https://docs.cloud.google.com/gemini/docs/conversational-analytics-api/data-agent-authored-context-databases#providing_context_with_querydata

# %% code
# Replace with your actual project and spanner details
spanner_ref_1 = geminidataanalytics.SpannerReference()
spanner_ref_1.database_reference = geminidataanalytics.SpannerDatabaseReference()
spanner_ref_1.database_reference.project_id = billing_project
spanner_ref_1.database_reference.region = "us-central1" # Example region
spanner_ref_1.database_reference.engine = engine
spanner_ref_1.database_reference.instance_id = spanner_instance_id
spanner_ref_1.database_reference.database_id = spanner_database_id
# optional set agent context reference
# spanner_ref_1.agent_context_reference.context_set_id = f"projects/{billing_project}/locations/{location}/contextSets/your_context_set_id"

datasource_references = geminidataanalytics.DatasourceReferences()
datasource_references.spanner_reference = spanner_ref_1

print("spanner DatasourceReferences created")

In [ ]:
# %% [markdown]
# ## 5. Stateful chat
#

# %% code
# Set up context for stateful chat
published_context = geminidataanalytics.Context()
published_context.system_instruction = system_instruction
published_context.datasource_references = datasource_references
# Optional: To enable advanced analysis with Python, include the following line:
published_context.options.analysis.python.enabled = True


In [ ]:
# %% [markdown]
# ## 6. Create a syncronize data agent
#

# %% code

data_agent = geminidataanalytics.DataAgent()
data_agent.data_analytics_agent.published_context = published_context
data_agent.name = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}" # Optional

request = geminidataanalytics.CreateDataAgentRequest(
    parent=f"projects/{billing_project}/locations/{location}",
    data_agent_id=data_agent_id, # Optional
    data_agent=data_agent,
)

try:
  response = data_agent_client.create_data_agent_sync(request=request)
  print("Data Agent created")
  print(response)
except Exception as e:
  print(f"Error creating Data Agent: {e}")


In [ ]:
# %% [markdown]
# ## 7. Create a asyncronize data agent
#

# data_agent_id = "data_agent_spanner"

#  data_agent = geminidataanalytics.DataAgent()
#  data_agent.data_analytics_agent.published_context = published_context
#  data_agent.name = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}" # Optional

#  request = geminidataanalytics.CreateDataAgentRequest(
#    parent=f"projects/{billing_project}/locations/{location}",
#    data_agent_id=data_agent_id, # Optional
#    data_agent=data_agent,
#  )

#  try:
#    data_agent_client.create_data_agent(request=request)
#    print("Data Agent created")
#  except Exception as e:
#    print(f"Error creating Data Agent: {e}")



In [ ]:
# %% [markdown]
# ## 8. Create a conversation
#

# %% code
# Initialize request arguments
conversation_id = "conversation_spanner_example"

conversation = geminidataanalytics.Conversation()
conversation.agents = [f'projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}']
conversation.name = f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}"

request = geminidataanalytics.CreateConversationRequest(
  parent=f"projects/{billing_project}/locations/{location}",
  conversation_id=conversation_id,
  conversation=conversation,
)

# Make the request
response = data_chat_client.create_conversation(request=request)

# Handle the response
print(response)


In [ ]:
# %% [markdown]
# ## 9. get a data agent
#

# %% code

# Initialize request arguments
request = geminidataanalytics.GetDataAgentRequest(
  name=f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",
)

# Make the request
response = data_agent_client.get_data_agent(request=request)

# Handle the response
print(response)


In [ ]:
# %% [markdown]
# ## 10. list data agent
#

# %% code
creator_filter = "YOUR-CREATOR-FILTER" #optional

request = geminidataanalytics.ListDataAgentsRequest(
  parent=f"projects/{billing_project}/locations/{location}",
  # creator_filter=creator_filter, optional
)

# Make the request
page_result = data_agent_client.list_data_agents(request=request)

# Handle the response
for response in page_result:
  print(response)


In [ ]:
# %% [markdown]
# ## 11. update a agent
#

# %% code

data_agent = geminidataanalytics.DataAgent()
data_agent.data_analytics_agent.published_context = published_context
data_agent.name = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
data_agent.description = "Updated description of the data agent."

update_mask = field_mask_pb2.FieldMask(paths=['description', 'data_analytics_agent.published_context'])

request = geminidataanalytics.UpdateDataAgentRequest(
  data_agent=data_agent,
  update_mask=update_mask,
)

try:
  # Make the request
  response = data_agent_client.update_data_agent_sync(request=request)
  print("Data Agent Updated")
  print(response)
except Exception as e:
  print(f"Error updating Data Agent: {e}")


In [ ]:

# %% [markdown]
# ## 12. Get IAM policy
#

# %% code

resource = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
request = iam_policy_pb2.GetIamPolicyRequest(
      resource=resource,
    )
try:
  response = data_agent_client.get_iam_policy(request=request)
  print("IAM Policy fetched successfully!")
  print(f"Response: {response}")
except Exception as e:
  print(f"Error setting IAM policy: {e}")


In [ ]:
# %% [markdown]
# ## 13. Update IAM policy
#

# %% code
role = "roles/geminidataanalytics.dataAgentEditor"
users = "tzha@google.com" # replace with your email

resource = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"

# Construct the IAM policy
binding = policy_pb2.Binding(
  role=role,
  members= [f"user:{i.strip()}" for i in users.split(",")]
)

policy = policy_pb2.Policy(bindings=[binding])

# Create the request
request = iam_policy_pb2.SetIamPolicyRequest(
  resource=resource,
  policy=policy
)

# Send the request
try:
  response = data_agent_client.set_iam_policy(request=request)
  print("IAM Policy set successfully!")
  print(f"Response: {response}")
except Exception as e:
  print(f"Error setting IAM policy: {e}")


In [ ]:
# %% [markdown]
# ## 14. Get a conversation
#

# %% code"

request = geminidataanalytics.GetConversationRequest(
  name = f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}"
)

# Make the request
response = data_chat_client.get_conversation(request=request)

# Handle the response
print(response)


In [ ]:
# %% [markdown]
# ## 15. list conversation
#

# %% code"
request = geminidataanalytics.ListConversationsRequest(
  parent=f"projects/{billing_project}/locations/{location}",
)

# Make the request
response = data_chat_client.list_conversations(request=request)

# Handle the response
print(response)


In [ ]:
# %% [markdown]
# ## 16. list messages in conversation
#

# %% code"
request = geminidataanalytics.ListMessagesRequest(
  parent=f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
)

# Make the request
response = data_chat_client.list_messages(request=request)

# Handle the response
print(response)


In [ ]:
# %% [markdown]
# ## 17. Stateful chat with agent and conversation
#

# %% code"
# Create a request that contains a single user message (your question)
question = "How many cards has John Avon illustrated in total?"
messages = [geminidataanalytics.Message()]
messages[0].user_message.text = question


# Create a conversation_reference
conversation_reference = geminidataanalytics.ConversationReference()
conversation_reference.conversation = f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}"
conversation_reference.data_agent_context.data_agent = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
# conversation_reference.data_agent_context.credentials = credentials

# Form the request
request = geminidataanalytics.ChatRequest(
    parent = f"projects/{billing_project}/locations/{location}",
    messages = messages,
    conversation_reference = conversation_reference
)

# Make the request
stream = data_chat_client.chat(request=request, timeout=300) #custom timeout up to 600s

# Handle the response
for response in stream:
    show_message(response)


In [ ]:
# %% [markdown]
# ## 18. Stateless chat without conversation
#

# %% code
# Create a request that contains a single user message (your question)
question = "How many cards has John Avon illustrated in total?"

messages = [geminidataanalytics.Message()]
messages[0].user_message.text = question


data_agent_context = geminidataanalytics.DataAgentContext()
data_agent_context.data_agent = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
# data_agent_context.credentials = credentials

# Form the request
request = geminidataanalytics.ChatRequest(
    parent=f"projects/{billing_project}/locations/{location}",
    messages=messages,
    data_agent_context = data_agent_context
)

# Make the request
stream = data_chat_client.chat(request=request, timeout=300) #custom timeout up to 600s

# Handle the response
for response in stream:
    show_message(response)


In [ ]:
# %% [markdown]
# ## 19. Ask questions with inline context
#

# %% code
# Create a request that contains a single user message (your question)
messages = [geminidataanalytics.Message()]
messages[0].user_message.text = question

inline_context = geminidataanalytics.Context()
inline_context.system_instruction = system_instruction
inline_context.datasource_references = datasource_references
# Optional: To enable advanced analysis with Python, include the following line:
inline_context.options.analysis.python.enabled = True

request = geminidataanalytics.ChatRequest(
    inline_context=inline_context,
    parent=f"projects/{billing_project}/locations/{location}",
    messages=messages,
)

# Make the request
stream = data_chat_client.chat(request=request, timeout=300) #custom timeout up to 600s

# Handle the response
for response in stream:
    show_message(response)


In [ ]:
# %% [markdown]
# ## 20. multi turn conversation
#

# %% code
# List that is used to track previous turns and is reused across requests
conversation_messages = []


# Use data agent context
data_agent_context = geminidataanalytics.DataAgentContext()
data_agent_context.data_agent = f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
# data_agent_context.credentials = credentials

# Helper function for calling the API
def multi_turn_Conversation(msg):

  message = geminidataanalytics.Message()
  message.user_message.text = msg

  # Send a multi-turn request by including previous turns and the new message
  conversation_messages.append(message)

  request = geminidataanalytics.ChatRequest(
    parent=f"projects/{billing_project}/locations/{location}",
    messages=conversation_messages,
    # Use data agent context
    data_agent_context=data_agent_context,
    # Use inline context
    # inline_context=inline_context,
  )

  # Make the request
  stream = data_chat_client.chat(request=request, timeout=300) #custom timeout up to 600s

  # Handle the response
  for response in stream:
    show_message(response)
    conversation_messages.append(response)

# Send the first turn request
multi_turn_Conversation("How many cards has John Avon illustrated in total?")

# Send follow-up turn request
multi_turn_Conversation("Which artist has the most cards?")


In [ ]:
# %% [markdown]
# ## 21. Delete agent and conversation
#

# %% code
request = geminidataanalytics.DeleteDataAgentRequest(
  name=f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",
)

try:
  # Make the request
  data_agent_client.delete_data_agent_sync(request=request)
  print("Data Agent Deleted")
except Exception as e:
  print(f"Error deleting Data Agent: {e}")



request = geminidataanalytics.DeleteConversationRequest(
  name = f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}"
)

# Make the request
response = data_chat_client.delete_conversation(request=request)

# Handle the response
print(response)
